<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/Model12_Reference_Inspired_Feature_Enhancement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"

MODEL09_DATA_PATH = (
    RESULTS_PATH +
    "model10_recommended_features.parquet"
)

MODEL11_BENCHMARK_ROC = 0.7882
MODEL11_BENCHMARK_PR = 0.2905

os.makedirs(
    RESULTS_PATH,
    exist_ok=True
)

print("Raw data path:", DATA_PATH)
print("Model09 data:", MODEL09_DATA_PATH)


Raw data path: /content/drive/MyDrive/datasets/raw/
Model09 data: /content/drive/MyDrive/RupeeRisk/model10_recommended_features.parquet


In [ ]:
model09_df = pd.read_parquet(
    MODEL09_DATA_PATH
)

print(
    "Model09 dataset shape:",
    model09_df.shape
)

print(
    "Model09 feature count:",
    model09_df.shape[1] - 1
)

print(
    "Target present:",
    "TARGET" in model09_df.columns
)

display(
    model09_df.head()
)

Model09 dataset shape: (307511, 304)
Model09 feature count: 303
Target present: True


,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,CC_CC_DRAWING_LIMIT_RATIO_MAX,CC_CC_DRAWING_LIMIT_RATIO_MEAN,CC_CC_DRAWING_LIMIT_RATIO_SUM,CC_CC_MIN_PAYMENT_RATIO_MIN,CC_CC_MIN_PAYMENT_RATIO_MEAN,CC_CC_MIN_PAYMENT_RATIO_SUM,CC_CARD_COUNT,CC_LATE_RATE,CC_SEVERE_DPD_RATE,HAS_CREDIT_CARD_HISTORY
0,1,Cash loans,M,N,Y,0,202500.0,24700.5,351000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,0,Cash loans,F,N,N,0,270000.0,35698.5,1129500.0,Family,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,0,Revolving loans,M,Y,Y,0,67500.0,6750.0,135000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,0,Cash loans,F,N,Y,0,135000.0,29686.5,297000.0,Unaccompanied,...,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.0,1
4,0,Cash loans,M,N,Y,0,121500.0,21865.5,513000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [ ]:
if "TARGET" not in model09_df.columns:
    raise ValueError(
        "TARGET column not found."
    )

X_base = model09_df.drop(
    columns=["TARGET"]
).copy()

y = model09_df["TARGET"].copy()

print(
    "Base X shape:",
    X_base.shape
)

print(
    "Target shape:",
    y.shape
)

Base X shape: (307511, 303)
Target shape: (307511,)


In [ ]:
def safe_ratio(
    numerator,
    denominator
):
    denominator = denominator.replace(
        0,
        np.nan
    )

    result = (
        numerator /
        denominator
    )

    return result.replace(
        [np.inf, -np.inf],
        np.nan
    )


In [ ]:
application_raw = pd.read_csv(
    DATA_PATH +
    "application_train.csv"
)

application_raw["DAYS_EMPLOYED_ANOM"] = (
    application_raw["DAYS_EMPLOYED"] == 365243
).astype("int8")

application_raw["DAYS_EMPLOYED"] = (
    application_raw["DAYS_EMPLOYED"]
    .replace(365243, np.nan)
)

ext_cols = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

existing_ext = [
    c for c in ext_cols
    if c in application_raw.columns
]


# EXT aggregates
application_raw["REF_EXT_MEAN"] = (
    application_raw[existing_ext]
    .mean(axis=1)
)

application_raw["REF_EXT_STD"] = (
    application_raw[existing_ext]
    .std(axis=1)
)

application_raw["REF_EXT_MIN"] = (
    application_raw[existing_ext]
    .min(axis=1)
)

application_raw["REF_EXT_MAX"] = (
    application_raw[existing_ext]
    .max(axis=1)
)

application_raw["REF_EXT_RANGE"] = (
    application_raw["REF_EXT_MAX"] -
    application_raw["REF_EXT_MIN"]
)

application_raw["REF_EXT_NANCOUNT"] = (
    application_raw[existing_ext]
    .isna()
    .sum(axis=1)
)


# Pairwise EXT interactions
if all(
    c in application_raw.columns
    for c in [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2"
    ]
):
    application_raw["REF_EXT_S1_S2"] = (
        application_raw["EXT_SOURCE_1"] *
        application_raw["EXT_SOURCE_2"]
    )

if all(
    c in application_raw.columns
    for c in [
        "EXT_SOURCE_1",
        "EXT_SOURCE_3"
    ]
):
    application_raw["REF_EXT_S1_S3"] = (
        application_raw["EXT_SOURCE_1"] *
        application_raw["EXT_SOURCE_3"]
    )

if all(
    c in application_raw.columns
    for c in [
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]
):
    application_raw["REF_EXT_S2_S3"] = (
        application_raw["EXT_SOURCE_2"] *
        application_raw["EXT_SOURCE_3"]
    )


# Triple interaction
if all(
    c in application_raw.columns
    for c in ext_cols
):
    application_raw["REF_EXT_PRODUCT"] = (
        application_raw["EXT_SOURCE_1"] *
        application_raw["EXT_SOURCE_2"] *
        application_raw["EXT_SOURCE_3"]
    )


# EXT squares
for i in [1, 2, 3]:

    col = f"EXT_SOURCE_{i}"

    if col in application_raw.columns:

        application_raw[
            f"REF_{col}_SQ"
        ] = (
            application_raw[col] ** 2
        )


# Financial ratios
application_raw["REF_CREDIT_INCOME"] = safe_ratio(
    application_raw["AMT_CREDIT"],
    application_raw["AMT_INCOME_TOTAL"]
)

application_raw["REF_ANNUITY_INCOME"] = safe_ratio(
    application_raw["AMT_ANNUITY"],
    application_raw["AMT_INCOME_TOTAL"]
)

application_raw["REF_CREDIT_ANNUITY"] = safe_ratio(
    application_raw["AMT_CREDIT"],
    application_raw["AMT_ANNUITY"]
)

application_raw["REF_ANNUITY_CREDIT"] = safe_ratio(
    application_raw["AMT_ANNUITY"],
    application_raw["AMT_CREDIT"]
)

application_raw["REF_GOODS_INCOME"] = safe_ratio(
    application_raw["AMT_GOODS_PRICE"],
    application_raw["AMT_INCOME_TOTAL"]
)

application_raw["REF_CREDIT_GOODS"] = safe_ratio(
    application_raw["AMT_CREDIT"],
    application_raw["AMT_GOODS_PRICE"]
)

application_raw["REF_PAYMENT_LENGTH"] = safe_ratio(
    application_raw["AMT_CREDIT"],
    application_raw["AMT_ANNUITY"]
)


# Age / employment
application_raw["REF_AGE_YEARS"] = (
    application_raw["DAYS_BIRTH"] /
    -365.25
)

application_raw["REF_EMPLOYED_YEARS"] = safe_ratio(
    application_raw["DAYS_EMPLOYED"],
    pd.Series(-365.25, index=application_raw.index)
)

application_raw["REF_EMPLOYED_BIRTH_RATIO"] = safe_ratio(
    application_raw["DAYS_EMPLOYED"],
    application_raw["DAYS_BIRTH"]
)

application_raw["REF_ID_BIRTH_RATIO"] = safe_ratio(
    application_raw["DAYS_ID_PUBLISH"],
    application_raw["DAYS_BIRTH"]
)

application_raw["REF_REG_BIRTH_RATIO"] = safe_ratio(
    application_raw["DAYS_REGISTRATION"],
    application_raw["DAYS_BIRTH"]
)

application_raw["REF_PHONE_BIRTH_RATIO"] = safe_ratio(
    application_raw["DAYS_LAST_PHONE_CHANGE"],
    application_raw["DAYS_BIRTH"]
)


# Household / income
application_raw["REF_INCOME_PER_CHILD"] = safe_ratio(
    application_raw["AMT_INCOME_TOTAL"],
    application_raw["CNT_CHILDREN"] + 1
)

application_raw["REF_INCOME_PER_FAMILY"] = safe_ratio(
    application_raw["AMT_INCOME_TOTAL"],
    application_raw["CNT_FAM_MEMBERS"] + 1
)

application_raw["REF_CREDIT_PER_PERSON"] = safe_ratio(
    application_raw["AMT_CREDIT"],
    application_raw["CNT_FAM_MEMBERS"] + 1
)

application_raw["REF_ANNUITY_PER_PERSON"] = safe_ratio(
    application_raw["AMT_ANNUITY"],
    application_raw["CNT_FAM_MEMBERS"] + 1
)


# Social-circle risk
if all(
    c in application_raw.columns
    for c in [
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "OBS_30_CNT_SOCIAL_CIRCLE"
    ]
):
    application_raw["REF_DEF30_RATIO"] = safe_ratio(
        application_raw["DEF_30_CNT_SOCIAL_CIRCLE"],
        application_raw["OBS_30_CNT_SOCIAL_CIRCLE"]
    )

if all(
    c in application_raw.columns
    for c in [
        "DEF_60_CNT_SOCIAL_CIRCLE",
        "OBS_60_CNT_SOCIAL_CIRCLE"
    ]
):
    application_raw["REF_DEF60_RATIO"] = safe_ratio(
        application_raw["DEF_60_CNT_SOCIAL_CIRCLE"],
        application_raw["OBS_60_CNT_SOCIAL_CIRCLE"]
    )


# Document count
doc_cols = [
    c
    for c in application_raw.columns
    if c.startswith("FLAG_DOCUMENT_")
]

application_raw["REF_DOCUMENT_COUNT"] = (
    application_raw[doc_cols]
    .sum(axis=1)
)


# Application missingness
application_raw["REF_APP_NULL_COUNT"] = (
    application_raw.isna()
    .sum(axis=1)
)


application_ref_cols = [
    c for c in application_raw.columns
    if c.startswith("REF_")
]

application_features = (
    application_raw[
        ["SK_ID_CURR"] +
        application_ref_cols
    ]
    .copy()
)

print(
    "Application reference-inspired features:",
    len(application_ref_cols)
)

display(
    application_features.head()
)

del application_raw

gc.collect()

Application reference-inspired features: 34


,SK_ID_CURR,REF_EXT_MEAN,REF_EXT_STD,REF_EXT_MIN,REF_EXT_MAX,REF_EXT_RANGE,REF_EXT_NANCOUNT,REF_EXT_S1_S2,REF_EXT_S1_S3,REF_EXT_S2_S3,...,REF_REG_BIRTH_RATIO,REF_PHONE_BIRTH_RATIO,REF_INCOME_PER_CHILD,REF_INCOME_PER_FAMILY,REF_CREDIT_PER_PERSON,REF_ANNUITY_PER_PERSON,REF_DEF30_RATIO,REF_DEF60_RATIO,REF_DOCUMENT_COUNT,REF_APP_NULL_COUNT
0,100002,0.161787,0.092026,0.083037,0.262949,0.179912,0,0.021834,0.011573,0.036649,...,0.385583,0.119860,202500.0,101250.0,203298.75,12350.25,1.0,1.0,1,1
1,100003,0.466757,0.219895,0.311267,0.622246,0.310978,1,0.193685,NaN,NaN,...,0.070743,0.049389,270000.0,90000.0,431167.50,11899.50,0.0,0.0,1,6
2,100004,0.642739,0.122792,0.555912,0.729567,0.173655,1,NaN,NaN,0.405575,...,0.223669,0.042791,67500.0,33750.0,67500.00,3375.00,NaN,NaN,0,54
3,100006,0.650442,NaN,0.650442,0.650442,0.000000,2,NaN,NaN,NaN,...,0.517390,0.032465,135000.0,45000.0,104227.50,9895.50,0.0,0.0,1,63
4,100007,0.322738,NaN,0.322738,0.322738,0.000000,2,NaN,NaN,NaN,...,0.216285,0.055489,121500.0,60750.0,256500.00,10932.75,NaN,NaN,1,59


0

In [ ]:
# ============================================================
# CELL 7 — ADD APPLICATION ENHANCEMENTS
# ============================================================

application_features = (
    application_features
    .set_index("SK_ID_CURR")
)

# Get applicant IDs in the same order as model09_df
app_ids = pd.read_csv(
    DATA_PATH + "application_train.csv",
    usecols=["SK_ID_CURR"]
)

# Align application features to the Model09 row order
application_features_aligned = (
    application_features
    .reindex(app_ids["SK_ID_CURR"].values)
)

# Reset index so rows align exactly with X_base
application_features_aligned = (
    application_features_aligned
    .reset_index(drop=True)
)

application_features_aligned.index = X_base.index

# Add only the new REF_ features
X_enhanced = pd.concat(
    [
        X_base,
        application_features_aligned
    ],
    axis=1
)

print(
    "Shape after application enhancements:",
    X_enhanced.shape
)

print(
    "New application features added:",
    len(application_features_aligned.columns)
)

display(
    X_enhanced.head()
)

del (
    application_features,
    application_features_aligned,
    app_ids
)

gc.collect()

Shape after application enhancements: (307511, 337)
New application features added: 34


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,...,REF_REG_BIRTH_RATIO,REF_PHONE_BIRTH_RATIO,REF_INCOME_PER_CHILD,REF_INCOME_PER_FAMILY,REF_CREDIT_PER_PERSON,REF_ANNUITY_PER_PERSON,REF_DEF30_RATIO,REF_DEF60_RATIO,REF_DOCUMENT_COUNT,REF_APP_NULL_COUNT
0,Cash loans,M,N,Y,0,202500.0,24700.5,351000.0,Unaccompanied,Working,...,0.385583,0.119860,202500.0,101250.0,203298.75,12350.25,1.0,1.0,1,1
1,Cash loans,F,N,N,0,270000.0,35698.5,1129500.0,Family,State servant,...,0.070743,0.049389,270000.0,90000.0,431167.50,11899.50,0.0,0.0,1,6
2,Revolving loans,M,Y,Y,0,67500.0,6750.0,135000.0,Unaccompanied,Working,...,0.223669,0.042791,67500.0,33750.0,67500.00,3375.00,NaN,NaN,0,54
3,Cash loans,F,N,Y,0,135000.0,29686.5,297000.0,Unaccompanied,Working,...,0.517390,0.032465,135000.0,45000.0,104227.50,9895.50,0.0,0.0,1,63
4,Cash loans,M,N,Y,0,121500.0,21865.5,513000.0,Unaccompanied,Working,...,0.216285,0.055489,121500.0,60750.0,256500.00,10932.75,NaN,NaN,1,59


0

In [ ]:
bureau = pd.read_csv(
    DATA_PATH +
    "bureau.csv"
)

# Sentinel cleanup
for col in [
    "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT",
    "DAYS_CREDIT_UPDATE"
]:

    if col in bureau.columns:
        bureau[col] = (
            bureau[col]
            .replace(365243, np.nan)
        )


# Reference-inspired row-level ratios
bureau["REF_DEBT_CREDIT_RATIO"] = safe_ratio(
    bureau["AMT_CREDIT_SUM_DEBT"],
    bureau["AMT_CREDIT_SUM"]
)

bureau["REF_OVERDUE_DEBT_RATIO"] = safe_ratio(
    bureau["AMT_CREDIT_SUM_OVERDUE"],
    bureau["AMT_CREDIT_SUM_DEBT"]
)

bureau["REF_CREDIT_OVERDUE_RATIO"] = safe_ratio(
    bureau["AMT_CREDIT_SUM_OVERDUE"],
    bureau["AMT_CREDIT_SUM"]
)

bureau["REF_CREDIT_DURATION"] = (
    bureau["DAYS_CREDIT_ENDDATE"] -
    bureau["DAYS_CREDIT"]
)

bureau["REF_ENDDATE_DIFF"] = (
    bureau["DAYS_CREDIT_ENDDATE"] -
    bureau["DAYS_ENDDATE_FACT"]
)

bureau["REF_UPDATE_DIFF"] = (
    bureau["DAYS_CREDIT_UPDATE"] -
    bureau["DAYS_CREDIT"]
)


# Overall aggregates
bureau_overall = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        B12_DEBT_CREDIT_MEAN=(
            "REF_DEBT_CREDIT_RATIO",
            "mean"
        ),
        B12_DEBT_CREDIT_MAX=(
            "REF_DEBT_CREDIT_RATIO",
            "max"
        ),
        B12_OVERDUE_DEBT_MAX=(
            "REF_OVERDUE_DEBT_RATIO",
            "max"
        ),
        B12_CREDIT_DURATION_MEAN=(
            "REF_CREDIT_DURATION",
            "mean"
        ),
        B12_CREDIT_DURATION_MAX=(
            "REF_CREDIT_DURATION",
            "max"
        ),
        B12_UPDATE_DIFF_MEAN=(
            "REF_UPDATE_DIFF",
            "mean"
        ),
        B12_CREDIT_OVERDUE_RATIO_MAX=(
            "REF_CREDIT_OVERDUE_RATIO",
            "max"
        )
    )
    .reset_index()
)


# Active / Closed conditional aggregates
conditional_blocks = []

for status in [
    "Active",
    "Closed"
]:

    sub = bureau[
        bureau["CREDIT_ACTIVE"] == status
    ]

    if len(sub) == 0:
        continue

    block = (
        sub.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_{status.upper()}_COUNT":
                    ("SK_ID_BUREAU", "count"),

                f"B12_{status.upper()}_CREDIT_MEAN":
                    ("AMT_CREDIT_SUM", "mean"),

                f"B12_{status.upper()}_CREDIT_MAX":
                    ("AMT_CREDIT_SUM", "max"),

                f"B12_{status.upper()}_DEBT_MEAN":
                    ("AMT_CREDIT_SUM_DEBT", "mean"),

                f"B12_{status.upper()}_DEBT_MAX":
                    ("AMT_CREDIT_SUM_DEBT", "max"),

                f"B12_{status.upper()}_DEBT_CREDIT_MEAN":
                    ("REF_DEBT_CREDIT_RATIO", "mean")
            }
        )
        .reset_index()
    )

    conditional_blocks.append(
        block
    )


# Bureau recent windows
recent_blocks = []

for days, label in [
    (-180, "6M"),
    (-365, "1Y"),
    (-730, "2Y"),
    (-1095, "3Y")
]:

    sub = bureau[
        bureau["DAYS_CREDIT"] >= days
    ]

    if len(sub) == 0:
        continue

    block = (
        sub.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_BURO_{label}_COUNT":
                    ("SK_ID_BUREAU", "count"),

                f"B12_BURO_{label}_CREDIT_MEAN":
                    ("AMT_CREDIT_SUM", "mean"),

                f"B12_BURO_{label}_DEBT_MEAN":
                    ("AMT_CREDIT_SUM_DEBT", "mean"),

                f"B12_BURO_{label}_OVERDUE_MAX":
                    ("CREDIT_DAY_OVERDUE", "max"),

                f"B12_BURO_{label}_DEBT_RATIO_MEAN":
                    ("REF_DEBT_CREDIT_RATIO", "mean")
            }
        )
        .reset_index()
    )

    recent_blocks.append(
        block
    )


# Latest bureau record
bureau_sorted = bureau.sort_values(
    "DAYS_CREDIT",
    ascending=False
)

last_bureau = (
    bureau_sorted
    .groupby("SK_ID_CURR")
    .first()
    .reset_index()
)

last_bureau_features = (
    last_bureau[
        [
            "SK_ID_CURR",
            "DAYS_CREDIT",
            "AMT_CREDIT_SUM",
            "AMT_CREDIT_SUM_DEBT",
            "REF_DEBT_CREDIT_RATIO",
            "CREDIT_DAY_OVERDUE"
        ]
    ]
    .rename(
        columns={
            "DAYS_CREDIT":
                "B12_LAST_DAYS_CREDIT",

            "AMT_CREDIT_SUM":
                "B12_LAST_CREDIT_SUM",

            "AMT_CREDIT_SUM_DEBT":
                "B12_LAST_DEBT",

            "REF_DEBT_CREDIT_RATIO":
                "B12_LAST_DEBT_CREDIT_RATIO",

            "CREDIT_DAY_OVERDUE":
                "B12_LAST_DPD"
        }
    )
)


# Merge bureau enhancement blocks
bureau_features = bureau_overall.copy()

for block in conditional_blocks:
    bureau_features = bureau_features.merge(
        block,
        on="SK_ID_CURR",
        how="left"
    )

for block in recent_blocks:
    bureau_features = bureau_features.merge(
        block,
        on="SK_ID_CURR",
        how="left"
    )

bureau_features = bureau_features.merge(
    last_bureau_features,
    on="SK_ID_CURR",
    how="left"
)

print(
    "Bureau enhancement features:",
    bureau_features.shape[1] - 1
)

display(
    bureau_features.head()
)

del (
    bureau,
    bureau_overall,
    conditional_blocks,
    recent_blocks,
    last_bureau,
    last_bureau_features
)

gc.collect()


Bureau enhancement features: 44


,SK_ID_CURR,B12_DEBT_CREDIT_MEAN,B12_DEBT_CREDIT_MAX,B12_OVERDUE_DEBT_MAX,B12_CREDIT_DURATION_MEAN,B12_CREDIT_DURATION_MAX,B12_UPDATE_DIFF_MEAN,B12_CREDIT_OVERDUE_RATIO_MAX,B12_ACTIVE_COUNT,B12_ACTIVE_CREDIT_MEAN,...,B12_BURO_3Y_COUNT,B12_BURO_3Y_CREDIT_MEAN,B12_BURO_3Y_DEBT_MEAN,B12_BURO_3Y_OVERDUE_MAX,B12_BURO_3Y_DEBT_RATIO_MEAN,B12_LAST_DAYS_CREDIT,B12_LAST_CREDIT_SUM,B12_LAST_DEBT,B12_LAST_DEBT_CREDIT_RATIO,B12_LAST_DPD
0,100001,0.282518,0.987405,0.0,817.428571,1827.0,641.857143,0.0,3.0,294675.0000,...,6.0,227977.500,99447.75,0.0,0.329604,-49,378000.000,373239.0,0.987405,0
1,100002,0.136545,0.546180,0.0,719.833333,1822.0,374.125000,0.0,2.0,240994.2825,...,5.0,134044.713,61445.25,0.0,0.182060,-103,31988.565,0.0,0.000000,0
2,100003,0.000000,0.000000,NaN,856.250000,1822.0,584.750000,0.0,1.0,810000.0000,...,2.0,441326.250,0.00,0.0,0.000000,-606,810000.000,0.0,0.000000,0
3,100004,0.000000,0.000000,NaN,378.500000,731.0,335.000000,0.0,NaN,NaN,...,1.0,94537.800,0.00,0.0,0.000000,-408,94537.800,0.0,0.000000,0
4,100005,0.601256,0.954794,0.0,630.000000,1461.0,136.333333,0.0,2.0,299313.0000,...,3.0,219042.000,189469.50,0.0,0.601256,-62,29826.000,25321.5,0.848974,0


0

In [ ]:
# ============================================================
# FIX — RESTORE SK_ID_CURR TO X_ENHANCED
# ============================================================

app_ids = pd.read_csv(
    DATA_PATH + "application_train.csv",
    usecols=["SK_ID_CURR"]
)

# Safety check
print("X_enhanced rows:", len(X_enhanced))
print("Application ID rows:", len(app_ids))

if len(X_enhanced) != len(app_ids):
    raise ValueError(
        "Row count mismatch between X_enhanced and application_train."
    )

# Add applicant ID temporarily for relational merges
X_enhanced.insert(
    0,
    "SK_ID_CURR",
    app_ids["SK_ID_CURR"].values
)

print(
    "SK_ID_CURR added:",
    "SK_ID_CURR" in X_enhanced.columns
)

display(
    X_enhanced[["SK_ID_CURR"]].head()
)

X_enhanced rows: 307511
Application ID rows: 307511
SK_ID_CURR added: True


,SK_ID_CURR
0,100002
1,100003
2,100004
3,100006
4,100007


In [ ]:
prev = pd.read_csv(
    DATA_PATH +
    "previous_application.csv"
)

# Sentinel cleanup
for col in [
    c for c in prev.columns
    if "DAYS_" in c
]:

    prev[col] = (
        prev[col]
        .replace(365243, np.nan)
    )


# Reference-inspired ratios
prev["REF_APP_CREDIT_RATIO"] = safe_ratio(
    prev["AMT_APPLICATION"],
    prev["AMT_CREDIT"]
)

prev["REF_CREDIT_GOODS_RATIO"] = safe_ratio(
    prev["AMT_CREDIT"],
    prev["AMT_GOODS_PRICE"]
)

prev["REF_APP_GOODS_RATIO"] = safe_ratio(
    prev["AMT_APPLICATION"],
    prev["AMT_GOODS_PRICE"]
)

prev["REF_DOWN_PAYMENT_RATIO"] = safe_ratio(
    prev["AMT_DOWN_PAYMENT"],
    prev["AMT_CREDIT"]
)

prev["REF_INTEREST_AMOUNT"] = (
    prev["CNT_PAYMENT"] *
    prev["AMT_ANNUITY"] -
    prev["AMT_CREDIT"]
)

prev["REF_INTEREST_RATE"] = safe_ratio(
    prev["REF_INTEREST_AMOUNT"],
    prev["AMT_CREDIT"]
)


# Base applicant aggregates
prev_overall = (
    prev.groupby("SK_ID_CURR")
    .agg(
        B12_PREV_CREDIT_RATIO_MEAN=(
            "REF_APP_CREDIT_RATIO",
            "mean"
        ),

        B12_PREV_CREDIT_RATIO_MAX=(
            "REF_APP_CREDIT_RATIO",
            "max"
        ),

        B12_PREV_INTEREST_RATE_MEAN=(
            "REF_INTEREST_RATE",
            "mean"
        ),

        B12_PREV_INTEREST_RATE_MAX=(
            "REF_INTEREST_RATE",
            "max"
        ),

        B12_PREV_DOWN_RATIO_MEAN=(
            "REF_DOWN_PAYMENT_RATIO",
            "mean"
        ),

        B12_PREV_APP_GOODS_RATIO_MEAN=(
            "REF_APP_GOODS_RATIO",
            "mean"
        )
    )
    .reset_index()
)


# Recent previous-application windows
prev_recent_blocks = []

for days, label in [
    (-180, "6M"),
    (-365, "1Y"),
    (-730, "2Y"),
    (-1095, "3Y")
]:

    sub = prev[
        prev["DAYS_DECISION"] >= days
    ]

    if len(sub) == 0:
        continue

    block = (
        sub.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_PREV_{label}_COUNT":
                    ("SK_ID_PREV", "count"),

                f"B12_PREV_{label}_CREDIT_MEAN":
                    ("AMT_CREDIT", "mean"),

                f"B12_PREV_{label}_ANNUITY_MEAN":
                    ("AMT_ANNUITY", "mean"),

                f"B12_PREV_{label}_CREDIT_RATIO_MEAN":
                    ("REF_APP_CREDIT_RATIO", "mean"),

                f"B12_PREV_{label}_INTEREST_MEAN":
                    ("REF_INTEREST_RATE", "mean")
            }
        )
        .reset_index()
    )

    prev_recent_blocks.append(
        block
    )


# Latest previous application
prev_sorted = (
    prev.sort_values(
        "DAYS_DECISION",
        ascending=False
    )
)

last_prev = (
    prev_sorted
    .groupby("SK_ID_CURR")
    .first()
    .reset_index()
)

last_prev_features = (
    last_prev[
        [
            "SK_ID_CURR",
            "DAYS_DECISION",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "REF_APP_CREDIT_RATIO",
            "REF_INTEREST_RATE"
        ]
    ]
    .rename(
        columns={
            "DAYS_DECISION":
                "B12_PREV_LAST_DAYS_DECISION",

            "AMT_CREDIT":
                "B12_PREV_LAST_CREDIT",

            "AMT_ANNUITY":
                "B12_PREV_LAST_ANNUITY",

            "REF_APP_CREDIT_RATIO":
                "B12_PREV_LAST_CREDIT_RATIO",

            "REF_INTEREST_RATE":
                "B12_PREV_LAST_INTEREST_RATE"
        }
    )
)


prev_features = prev_overall.copy()

for block in prev_recent_blocks:

    prev_features = prev_features.merge(
        block,
        on="SK_ID_CURR",
        how="left"
    )

prev_features = prev_features.merge(
    last_prev_features,
    on="SK_ID_CURR",
    how="left"
)

print(
    "Previous-application enhancement features:",
    prev_features.shape[1] - 1
)

display(
    prev_features.head()
)

del (
    prev,
    prev_overall,
    prev_recent_blocks,
    last_prev,
    last_prev_features
)

gc.collect()

Previous-application enhancement features: 31


,SK_ID_CURR,B12_PREV_CREDIT_RATIO_MEAN,B12_PREV_CREDIT_RATIO_MAX,B12_PREV_INTEREST_RATE_MEAN,B12_PREV_INTEREST_RATE_MAX,B12_PREV_DOWN_RATIO_MEAN,B12_PREV_APP_GOODS_RATIO_MEAN,B12_PREV_6M_COUNT,B12_PREV_6M_CREDIT_MEAN,B12_PREV_6M_ANNUITY_MEAN,...,B12_PREV_3Y_COUNT,B12_PREV_3Y_CREDIT_MEAN,B12_PREV_3Y_ANNUITY_MEAN,B12_PREV_3Y_CREDIT_RATIO_MEAN,B12_PREV_3Y_INTEREST_MEAN,B12_PREV_LAST_DAYS_DECISION,B12_PREV_LAST_CREDIT,B12_PREV_LAST_ANNUITY,B12_PREV_LAST_CREDIT_RATIO,B12_PREV_LAST_INTEREST_RATE
0,100001,1.044079,1.044079,0.328793,0.328793,0.105940,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-1740,23787.0,3951.000,1.044079,0.328793
1,100002,1.000000,1.000000,0.240080,0.240080,0.000000,1.0,NaN,NaN,NaN,...,1.0,179055.00,9251.775,1.000000,0.240080,-606,179055.0,9251.775,1.000000,0.240080
2,100003,0.949329,1.011109,0.146201,0.188002,0.050585,1.0,NaN,NaN,NaN,...,2.0,692259.75,81462.330,0.918440,0.125300,-746,1035882.0,98356.995,0.868825,0.139400
3,100004,1.207699,1.207699,0.065801,0.065801,0.241719,1.0,NaN,NaN,NaN,...,1.0,20106.00,5357.250,1.207699,0.065801,-815,20106.0,5357.250,1.207699,0.065801
4,100005,1.111173,1.111173,0.438440,0.438440,0.111173,1.0,NaN,NaN,NaN,...,2.0,20076.75,4813.200,1.111173,0.438440,-315,0.0,4813.200,1.111173,0.438440


7

In [ ]:
X_enhanced = X_enhanced.merge(
    prev_features,
    on="SK_ID_CURR",
    how="left"
)

print(
    "Shape after previous-application enhancements:",
    X_enhanced.shape
)

del prev_features

gc.collect()

Shape after previous-application enhancements: (307511, 369)


0

In [ ]:
pos = pd.read_csv(
    DATA_PATH +
    "POS_CASH_balance.csv"
)

pos["REF_LATE_FLAG"] = (
    pos["SK_DPD"] > 0
).astype("int8")

pos["REF_DPD_RATIO"] = safe_ratio(
    pos["SK_DPD"],
    pos["SK_DPD_DEF"] + 1
)


# Recent POS windows
pos_blocks = []

for months, label in [
    (-6, "6M"),
    (-12, "1Y"),
    (-24, "2Y")
]:

    sub = pos[
        pos["MONTHS_BALANCE"] >= months
    ]

    if len(sub) == 0:
        continue

    block = (
        sub.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_POS_{label}_COUNT":
                    ("SK_ID_PREV", "count"),

                f"B12_POS_{label}_DPD_MEAN":
                    ("SK_DPD", "mean"),

                f"B12_POS_{label}_DPD_MAX":
                    ("SK_DPD", "max"),

                f"B12_POS_{label}_LATE_RATE":
                    ("REF_LATE_FLAG", "mean")
            }
        )
        .reset_index()
    )

    pos_blocks.append(
        block
    )


# Latest POS record
pos_sorted = pos.sort_values(
    "MONTHS_BALANCE",
    ascending=False
)

last_pos = (
    pos_sorted
    .groupby("SK_ID_CURR")
    .first()
    .reset_index()
)

last_pos_features = (
    last_pos[
        [
            "SK_ID_CURR",
            "MONTHS_BALANCE",
            "SK_DPD",
            "SK_DPD_DEF",
            "REF_LATE_FLAG"
        ]
    ]
    .rename(
        columns={
            "MONTHS_BALANCE":
                "B12_POS_LAST_MONTH",

            "SK_DPD":
                "B12_POS_LAST_DPD",

            "SK_DPD_DEF":
                "B12_POS_LAST_DPD_DEF",

            "REF_LATE_FLAG":
                "B12_POS_LAST_LATE"
        }
    )
)

pos_features = (
    last_pos_features.copy()
)

for block in pos_blocks:

    pos_features = pos_features.merge(
        block,
        on="SK_ID_CURR",
        how="outer"
    )

print(
    "POS enhancement features:",
    pos_features.shape[1] - 1
)

display(
    pos_features.head()
)

del (
    pos,
    pos_blocks,
    last_pos,
    last_pos_features
)

gc.collect()

POS enhancement features: 16


,SK_ID_CURR,B12_POS_LAST_MONTH,B12_POS_LAST_DPD,B12_POS_LAST_DPD_DEF,B12_POS_LAST_LATE,B12_POS_6M_COUNT,B12_POS_6M_DPD_MEAN,B12_POS_6M_DPD_MAX,B12_POS_6M_LATE_RATE,B12_POS_1Y_COUNT,B12_POS_1Y_DPD_MEAN,B12_POS_1Y_DPD_MAX,B12_POS_1Y_LATE_RATE,B12_POS_2Y_COUNT,B12_POS_2Y_DPD_MEAN,B12_POS_2Y_DPD_MAX,B12_POS_2Y_LATE_RATE
0,100001,-53,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,-1,0,0,0,6.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,19.0,0.0,0.0,0.0
2,100003,-18,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,0.0,0.0,0.0
3,100004,-24,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0
4,100005,-15,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.0,0.0,0.0


0

In [ ]:
X_enhanced = X_enhanced.merge(
    pos_features,
    on="SK_ID_CURR",
    how="left"
)

del pos_features

gc.collect()

print(
    "Shape after POS enhancements:",
    X_enhanced.shape
)



Shape after POS enhancements: (307511, 385)


In [ ]:
ins = pd.read_csv(
    DATA_PATH +
    "installments_payments.csv"
)

# Payment ratio
ins["REF_PAYMENT_RATIO"] = safe_ratio(
    ins["AMT_PAYMENT"],
    ins["AMT_INSTALMENT"]
)

# Payment difference
ins["REF_PAYMENT_DIFF"] = (
    ins["AMT_INSTALMENT"] -
    ins["AMT_PAYMENT"]
)

# Delay
ins["REF_PAYMENT_DELAY"] = (
    ins["DAYS_ENTRY_PAYMENT"] -
    ins["DAYS_INSTALMENT"]
)

ins["REF_LATE_PAYMENT"] = (
    ins["REF_PAYMENT_DELAY"] > 0
).astype("int8")

ins["REF_UNDERPAYMENT"] = (
    ins["REF_PAYMENT_DIFF"] > 0
).astype("int8")


# Overall installment behavior
ins_overall = (
    ins.groupby("SK_ID_CURR")
    .agg(
        B12_INS_PAYMENT_RATIO_MEAN=(
            "REF_PAYMENT_RATIO",
            "mean"
        ),

        B12_INS_PAYMENT_RATIO_MAX=(
            "REF_PAYMENT_RATIO",
            "max"
        ),

        B12_INS_PAYMENT_DIFF_MEAN=(
            "REF_PAYMENT_DIFF",
            "mean"
        ),

        B12_INS_PAYMENT_DELAY_MEAN=(
            "REF_PAYMENT_DELAY",
            "mean"
        ),

        B12_INS_PAYMENT_DELAY_MAX=(
            "REF_PAYMENT_DELAY",
            "max"
        ),

        B12_INS_LATE_RATE=(
            "REF_LATE_PAYMENT",
            "mean"
        ),

        B12_INS_UNDERPAY_RATE=(
            "REF_UNDERPAYMENT",
            "mean"
        )
    )
    .reset_index()
)


# Reference-inspired last-K installment behavior
ins_sorted = (
    ins.sort_values(
        "DAYS_INSTALMENT",
        ascending=False
    )
)

last_k_blocks = []

for k in [
    3,
    5,
    10,
    30
]:

    last_k = (
        ins_sorted
        .groupby("SK_ID_CURR")
        .head(k)
    )

    block = (
        last_k.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_INS_LAST{k}_DPD_MEAN":
                    ("REF_PAYMENT_DELAY", "mean"),

                f"B12_INS_LAST{k}_DPD_MAX":
                    ("REF_PAYMENT_DELAY", "max"),

                f"B12_INS_LAST{k}_PAYRATIO_MEAN":
                    ("REF_PAYMENT_RATIO", "mean"),

                f"B12_INS_LAST{k}_PAYDIFF_MEAN":
                    ("REF_PAYMENT_DIFF", "mean"),

                f"B12_INS_LAST{k}_LATE_RATE":
                    ("REF_LATE_PAYMENT", "mean")
            }
        )
        .reset_index()
    )

    last_k_blocks.append(
        block
    )


# Recent 6M / 1Y installment behavior
ins_recent_blocks = []

for days, label in [
    (-180, "6M"),
    (-365, "1Y")
]:

    sub = ins[
        ins["DAYS_INSTALMENT"] >= days
    ]

    if len(sub) == 0:
        continue

    block = (
        sub.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_INS_{label}_COUNT":
                    ("SK_ID_PREV", "count"),

                f"B12_INS_{label}_DELAY_MEAN":
                    ("REF_PAYMENT_DELAY", "mean"),

                f"B12_INS_{label}_LATE_RATE":
                    ("REF_LATE_PAYMENT", "mean"),

                f"B12_INS_{label}_PAYRATIO_MEAN":
                    ("REF_PAYMENT_RATIO", "mean")
            }
        )
        .reset_index()
    )

    ins_recent_blocks.append(
        block
    )


installment_features = (
    ins_overall.copy()
)

for block in last_k_blocks:

    installment_features = (
        installment_features
        .merge(
            block,
            on="SK_ID_CURR",
            how="left"
        )
    )

for block in ins_recent_blocks:

    installment_features = (
        installment_features
        .merge(
            block,
            on="SK_ID_CURR",
            how="left"
        )
    )

print(
    "Installment enhancement features:",
    installment_features.shape[1] - 1
)

display(
    installment_features.head()
)

del (
    ins,
    ins_overall,
    last_k_blocks,
    ins_recent_blocks,
    ins_sorted
)

gc.collect()


# ============================================================
# CELL 15 — MERGE INSTALLMENT FEATURES
# ============================================================

X_enhanced = X_enhanced.merge(
    installment_features,
    on="SK_ID_CURR",
    how="left"
)

del installment_features

gc.collect()

print(
    "Shape after installment enhancements:",
    X_enhanced.shape
)


Installment enhancement features: 35


,SK_ID_CURR,B12_INS_PAYMENT_RATIO_MEAN,B12_INS_PAYMENT_RATIO_MAX,B12_INS_PAYMENT_DIFF_MEAN,B12_INS_PAYMENT_DELAY_MEAN,B12_INS_PAYMENT_DELAY_MAX,B12_INS_LATE_RATE,B12_INS_UNDERPAY_RATE,B12_INS_LAST3_DPD_MEAN,B12_INS_LAST3_DPD_MAX,...,B12_INS_LAST30_PAYDIFF_MEAN,B12_INS_LAST30_LATE_RATE,B12_INS_6M_COUNT,B12_INS_6M_DELAY_MEAN,B12_INS_6M_LATE_RATE,B12_INS_6M_PAYRATIO_MEAN,B12_INS_1Y_COUNT,B12_INS_1Y_DELAY_MEAN,B12_INS_1Y_LATE_RATE,B12_INS_1Y_PAYRATIO_MEAN
0,100001,1.0,1.0,0.0,-7.285714,11.0,0.142857,0.0,-18.666667,-9.0,...,0.0,0.142857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,1.0,1.0,0.0,-20.421053,-12.0,0.000000,0.0,-16.666667,-12.0,...,0.0,0.000000,6.0,-17.0,0.0,1.0,12.0,-17.583333,0.0,1.0
2,100003,1.0,1.0,0.0,-7.160000,-1.0,0.000000,0.0,-5.333333,-4.0,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,1.0,1.0,0.0,-7.666667,-3.0,0.000000,0.0,-7.666667,-3.0,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,1.0,1.0,0.0,-23.555556,1.0,0.111111,0.0,-17.333333,-4.0,...,0.0,0.111111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Shape after installment enhancements: (307511, 420)


In [ ]:
cc = pd.read_csv(
    DATA_PATH +
    "credit_card_balance.csv"
)

# Utilization
cc["REF_UTILIZATION"] = safe_ratio(
    cc["AMT_BALANCE"],
    cc["AMT_CREDIT_LIMIT_ACTUAL"]
)

cc["REF_DPD_FLAG"] = (
    cc["SK_DPD"] > 0
).astype("int8")


# Overall card behavior
cc_overall = (
    cc.groupby("SK_ID_CURR")
    .agg(
        B12_CC_UTIL_MEAN=(
            "REF_UTILIZATION",
            "mean"
        ),

        B12_CC_UTIL_MAX=(
            "REF_UTILIZATION",
            "max"
        ),

        B12_CC_BALANCE_MEAN=(
            "AMT_BALANCE",
            "mean"
        ),

        B12_CC_BALANCE_MAX=(
            "AMT_BALANCE",
            "max"
        ),

        B12_CC_DPD_MAX=(
            "SK_DPD",
            "max"
        ),

        B12_CC_DPD_RATE=(
            "REF_DPD_FLAG",
            "mean"
        )
    )
    .reset_index()
)


# Recent credit-card windows
cc_recent_blocks = []

for months, label in [
    (-6, "6M"),
    (-12, "1Y"),
    (-24, "2Y")
]:

    sub = cc[
        cc["MONTHS_BALANCE"] >= months
    ]

    if len(sub) == 0:
        continue

    block = (
        sub.groupby("SK_ID_CURR")
        .agg(
            **{
                f"B12_CC_{label}_COUNT":
                    ("SK_ID_PREV", "count"),

                f"B12_CC_{label}_UTIL_MEAN":
                    ("REF_UTILIZATION", "mean"),

                f"B12_CC_{label}_BALANCE_MEAN":
                    ("AMT_BALANCE", "mean"),

                f"B12_CC_{label}_DPD_MAX":
                    ("SK_DPD", "max"),

                f"B12_CC_{label}_DPD_RATE":
                    ("REF_DPD_FLAG", "mean")
            }
        )
        .reset_index()
    )

    cc_recent_blocks.append(
        block
    )


# Latest card record
cc_sorted = cc.sort_values(
    "MONTHS_BALANCE",
    ascending=False
)

last_cc = (
    cc_sorted
    .groupby("SK_ID_CURR")
    .first()
    .reset_index()
)

last_cc_features = (
    last_cc[
        [
            "SK_ID_CURR",
            "MONTHS_BALANCE",
            "AMT_BALANCE",
            "AMT_CREDIT_LIMIT_ACTUAL",
            "REF_UTILIZATION",
            "SK_DPD"
        ]
    ]
    .rename(
        columns={
            "MONTHS_BALANCE":
                "B12_CC_LAST_MONTH",

            "AMT_BALANCE":
                "B12_CC_LAST_BALANCE",

            "AMT_CREDIT_LIMIT_ACTUAL":
                "B12_CC_LAST_LIMIT",

            "REF_UTILIZATION":
                "B12_CC_LAST_UTILIZATION",

            "SK_DPD":
                "B12_CC_LAST_DPD"
        }
    )
)


credit_card_features = (
    cc_overall.copy()
)

for block in cc_recent_blocks:

    credit_card_features = (
        credit_card_features
        .merge(
            block,
            on="SK_ID_CURR",
            how="left"
        )
    )

credit_card_features = (
    credit_card_features
    .merge(
        last_cc_features,
        on="SK_ID_CURR",
        how="left"
    )
)

print(
    "Credit-card enhancement features:",
    credit_card_features.shape[1] - 1
)

display(
    credit_card_features.head()
)

del (
    cc,
    cc_overall,
    cc_recent_blocks,
    last_cc,
    last_cc_features,
    cc_sorted
)

gc.collect()


# ============================================================
# CELL 17 — MERGE CREDIT CARD FEATURES
# ============================================================

X_enhanced = X_enhanced.merge(
    credit_card_features,
    on="SK_ID_CURR",
    how="left"
)

del credit_card_features

gc.collect()

print(
    "Shape after credit-card enhancements:",
    X_enhanced.shape
)



Credit-card enhancement features: 26


,SK_ID_CURR,B12_CC_UTIL_MEAN,B12_CC_UTIL_MAX,B12_CC_BALANCE_MEAN,B12_CC_BALANCE_MAX,B12_CC_DPD_MAX,B12_CC_DPD_RATE,B12_CC_6M_COUNT,B12_CC_6M_UTIL_MEAN,B12_CC_6M_BALANCE_MEAN,...,B12_CC_2Y_COUNT,B12_CC_2Y_UTIL_MEAN,B12_CC_2Y_BALANCE_MEAN,B12_CC_2Y_DPD_MAX,B12_CC_2Y_DPD_RATE,B12_CC_LAST_MONTH,B12_CC_LAST_BALANCE,B12_CC_LAST_LIMIT,B12_CC_LAST_UTILIZATION,B12_CC_LAST_DPD
0,100006,0.000000,0.00000,0.000000,0.00,0,0.000000,6.0,0.0,0.0,...,6,0.0,0.0,0,0.0,-1,0.0,270000,0.0,0
1,100011,0.302678,1.05000,54482.111149,189000.00,0,0.000000,5.0,0.0,0.0,...,23,0.0,0.0,0,0.0,-2,0.0,90000,0.0,0
2,100013,0.115301,1.02489,18159.919219,161420.22,1,0.010417,6.0,0.0,0.0,...,24,0.0,0.0,0,0.0,-1,0.0,45000,0.0,0
3,100021,0.000000,0.00000,0.000000,0.00,0,0.000000,5.0,0.0,0.0,...,17,0.0,0.0,0,0.0,-2,0.0,675000,0.0,0
4,100023,0.000000,0.00000,0.000000,0.00,0,0.000000,3.0,0.0,0.0,...,8,0.0,0.0,0,0.0,-4,0.0,225000,0.0,0


Shape after credit-card enhancements: (307511, 446)


In [ ]:
# SK_ID_CURR stays as the join key.
# Remove accidental duplicated feature names, if any.

duplicate_feature_names = (
    X_enhanced.columns[
        X_enhanced.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate column names:",
    len(duplicate_feature_names)
)

if duplicate_feature_names:

    print(
        duplicate_feature_names[:20]
    )

    X_enhanced = (
        X_enhanced.loc[
            :,
            ~X_enhanced.columns.duplicated()
        ]
    )



Duplicate column names: 0


In [ ]:
feature_columns = [
    c
    for c in X_enhanced.columns
    if c != "SK_ID_CURR"
]

print(
    "Final Model12 feature count:",
    len(feature_columns)
)

print(
    "Model09 feature count:",
    X_base.shape[1] - 1
)

print(
    "New features added:",
    len(feature_columns) - (
        X_base.shape[1] - 1
    )
)


# Infinity check
numeric_part = (
    X_enhanced
    .select_dtypes(
        include=np.number
    )
)

inf_count = (
    np.isinf(numeric_part)
    .sum()
    .sum()
)

print(
    "Infinite values:",
    inf_count
)


Final Model12 feature count: 445
Model09 feature count: 302
New features added: 143
Infinite values: 0


In [ ]:
X_enhanced = X_enhanced.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "NaN values:",
    X_enhanced.isna().sum().sum()
)


NaN values: 37601379


In [ ]:
model12_data = X_enhanced.copy()

model12_data["TARGET"] = y.values

MODEL12_DATA_PATH = (
    RESULTS_PATH +
    "model12_reference_inspired_features.parquet"
)

model12_data.to_parquet(
    MODEL12_DATA_PATH,
    index=False
)

print(
    "Model12 dataset saved to:",
    MODEL12_DATA_PATH
)

print(
    "Model12 shape:",
    model12_data.shape
)


# ============================================================
# CELL 22 — TRAIN / VALIDATION SPLIT
# ============================================================

X = model12_data.drop(
    columns=["TARGET"]
)

y = model12_data["TARGET"]

X_train_outer, X_valid_outer, y_train_outer, y_valid_outer = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print(
    "Outer training shape:",
    X_train_outer.shape
)

print(
    "Outer validation shape:",
    X_valid_outer.shape
)



Model12 dataset saved to: /content/drive/MyDrive/RupeeRisk/model12_reference_inspired_features.parquet
Model12 shape: (307511, 447)
Outer training shape: (246008, 446)
Outer validation shape: (61503, 446)


In [ ]:
del model12_data
del model09_df
del X_base

# These feature-engineering tables should also be gone if still present
for var_name in [
    "application_features",
    "bureau_features",
    "prev_features",
    "pos_features",
    "installment_features",
    "credit_card_features"
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

print("Memory cleanup complete.")
!free -h

Memory cleanup complete.
               total        used        free      shared  buff/cache   available
Mem:            12Gi       8.7Gi       2.3Gi       2.0Mi       1.6Gi       3.6Gi
Swap:             0B          0B          0B


In [ ]:
X_train_inner, X_es, y_train_inner, y_es = (
    train_test_split(
        X_train_outer,
        y_train_outer,
        test_size=0.15,
        random_state=42,
        stratify=y_train_outer
    )
)

print(
    "Inner training shape:",
    X_train_inner.shape
)

print(
    "Early-stopping validation shape:",
    X_es.shape
)


Inner training shape: (209106, 446)
Early-stopping validation shape: (36902, 446)


In [ ]:
X_train_inner, X_es, y_train_inner, y_es = (
    train_test_split(
        X_train_outer,
        y_train_outer,
        test_size=0.15,
        random_state=42,
        stratify=y_train_outer
    )
)

print(
    "Inner training shape:",
    X_train_inner.shape
)

print(
    "Early-stopping validation shape:",
    X_es.shape
)


Inner training shape: (209106, 446)
Early-stopping validation shape: (36902, 446)


In [ ]:
# ============================================================
# CELL 24 — IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ============================================================

numeric_features = (
    X_train_inner
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train_inner
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nFirst numeric features:")
print(numeric_features[:10])

print("\nCategorical features:")
print(categorical_features)

Numeric features: 430
Categorical features: 16

First numeric features:
['SK_ID_CURR', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE']

Categorical features:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)



In [ ]:
# ============================================================
# MEMORY CLEANUP BEFORE TRANSFORMATION
# ============================================================

# No longer needed after inner split
if "X_train_outer" in globals():
    del X_train_outer

if "y_train_outer" in globals():
    del y_train_outer

# Original unsplit matrices
if "X" in globals():
    del X

if "y" in globals():
    del y

# Old feature-engineering / source objects
for var_name in [
    "model12_data",
    "model09_df",
    "X_base",
    "application_features",
    "bureau_features",
    "prev_features",
    "pos_features",
    "installment_features",
    "credit_card_features"
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

print("Memory after cleanup:")
!free -h

Memory after cleanup:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       7.9Gi       3.1Gi       2.0Mi       1.6Gi       4.4Gi
Swap:             0B          0B          0B


In [ ]:
del X_valid_outer
gc.collect()

0

In [ ]:
print(
    "Fitting preprocessor..."
)

X_train_inner_transformed = (
    preprocessor.fit_transform(
        X_train_inner
    )
)

X_es_transformed = (
    preprocessor.transform(
        X_es
    )
)

X_valid_transformed = (
    preprocessor.transform(
        X_valid_outer
    )
)

print(
    "Transformed inner training:",
    X_train_inner_transformed.shape
)

print(
    "Transformed early stopping:",
    X_es_transformed.shape
)

print(
    "Transformed final validation:",
    X_valid_transformed.shape
)


# Release raw matrices
del (
    X_train_inner,
    X_es
)

gc.collect()



Fitting preprocessor...
Transformed inner training: (209106, 576)
Transformed early stopping: (36902, 576)
Transformed final validation: (61503, 576)


105

In [ ]:
reference_xgb = XGBClassifier(

    n_estimators=10000,

    learning_rate=0.01,

    max_depth=5,

    min_child_weight=40,

    subsample=0.8,

    colsample_bytree=0.3,

    gamma=0.1,

    reg_alpha=0.1,

    reg_lambda=1.0,

    objective="binary:logistic",

    eval_metric="auc",

    tree_method="hist",

    random_state=42,

    n_jobs=-1,

    early_stopping_rounds=200
)

print(
    "Reference-style XGBoost created."
)


Reference-style XGBoost created.


In [ ]:
del numeric_transformer
del categorical_transformer
del numeric_features
del categorical_features

gc.collect()

!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       8.5Gi       3.0Gi       2.0Mi       1.2Gi       3.9Gi
Swap:             0B          0B          0B


In [ ]:
print("Train transformed:", X_train_inner_transformed.shape)
print("ES transformed:", X_es_transformed.shape)
print("Valid transformed:", X_valid_transformed.shape)

!free -h

Train transformed: (209106, 576)
ES transformed: (36902, 576)
Valid transformed: (61503, 576)
               total        used        free      shared  buff/cache   available
Mem:            12Gi       8.5Gi       2.9Gi       2.0Mi       1.2Gi       3.9Gi
Swap:             0B          0B          0B


In [ ]:
print(
    "Training Model12 XGBoost..."
)

reference_xgb.fit(
    X_train_inner_transformed,
    y_train_inner,
    eval_set=[
        (
            X_es_transformed,
            y_es
        )
    ],
    verbose=500
)

print(
    "Model12 training complete."
)

print(
    "Best iteration:",
    reference_xgb.best_iteration
)



Training Model12 XGBoost...
[0]	validation_0-auc:0.72307
[500]	validation_0-auc:0.77444
[1000]	validation_0-auc:0.78191
[1500]	validation_0-auc:0.78516
[2000]	validation_0-auc:0.78715
[2500]	validation_0-auc:0.78838
[3000]	validation_0-auc:0.78918
[3500]	validation_0-auc:0.78988
[4000]	validation_0-auc:0.79044
[4500]	validation_0-auc:0.79077
[5000]	validation_0-auc:0.79104
[5378]	validation_0-auc:0.79103
Model12 training complete.
Best iteration: 5178


In [35]:
valid_proba = (
    reference_xgb
    .predict_proba(
        X_valid_transformed
    )[:, 1]
)

model12_roc_auc = roc_auc_score(
    y_valid_outer,
    valid_proba
)

model12_pr_auc = average_precision_score(
    y_valid_outer,
    valid_proba
)

print(
    "============================================================"
)

print(
    "MODEL12 FINAL VALIDATION"
)

print(
    f"ROC-AUC: {model12_roc_auc:.4f}"
)

print(
    f"PR-AUC:  {model12_pr_auc:.4f}"
)

print(
    "============================================================"
)


MODEL12 FINAL VALIDATION
ROC-AUC: 0.7942
PR-AUC:  0.2975


In [36]:
roc_change = (
    model12_roc_auc -
    MODEL11_BENCHMARK_ROC
)

pr_change = (
    model12_pr_auc -
    MODEL11_BENCHMARK_PR
)

print(
    "IMPROVEMENT OVER MODEL11"
)

print(
    f"ROC-AUC change: {roc_change:+.4f}"
)

print(
    f"PR-AUC change:  {pr_change:+.4f}"
)

print(
    "Distance from 0.80 ROC-AUC:",
    f"{0.8000 - model12_roc_auc:+.4f}"
)


IMPROVEMENT OVER MODEL11
ROC-AUC change: +0.0060
PR-AUC change:  +0.0070
Distance from 0.80 ROC-AUC: +0.0058


In [37]:
model12_results = pd.DataFrame({

    "Experiment": [
        "Model11 Optuna XGBoost",
        "Model12 Reference-Inspired"
    ],

    "ROC-AUC": [
        MODEL11_BENCHMARK_ROC,
        model12_roc_auc
    ],

    "PR-AUC": [
        MODEL11_BENCHMARK_PR,
        model12_pr_auc
    ],

    "ROC-AUC Change": [
        0.0,
        roc_change
    ],

    "PR-AUC Change": [
        0.0,
        pr_change
    ],

    "Feature Count": [
        X_base.shape[1] - 1,
        X.shape[1] - 1
    ],

    "Best Iteration": [
        np.nan,
        reference_xgb.best_iteration
    ]
})

display(
    model12_results.style.format({
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
        "ROC-AUC Change": "{:+.4f}",
        "PR-AUC Change": "{:+.4f}"
    })
)



NameError: name 'X_base' is not defined

In [38]:
MODEL12_RESULTS_PATH = (
    RESULTS_PATH +
    "model12_reference_inspired_results.csv"
)

model12_results.to_csv(
    MODEL12_RESULTS_PATH,
    index=False
)

print(
    "Results saved to:",
    MODEL12_RESULTS_PATH
)


NameError: name 'model12_results' is not defined

In [42]:
# ============================================================
# RECREATE MODEL12 RESULT SUMMARY
# ============================================================

MODEL12_BENCHMARK_ROC = 0.7882
MODEL12_BENCHMARK_PR = 0.2905

model12_results = pd.DataFrame({
    "Experiment": [
        "Model11 Optuna",
        "Model12 Reference-Inspired"
    ],

    "ROC-AUC": [
        MODEL12_BENCHMARK_ROC,
        model12_roc_auc
    ],

    "PR-AUC": [
        MODEL12_BENCHMARK_PR,
        model12_pr_auc
    ],

    "ROC-AUC Change": [
        0.0,
        model12_roc_auc - MODEL12_BENCHMARK_ROC
    ],

    "PR-AUC Change": [
        0.0,
        model12_pr_auc - MODEL12_BENCHMARK_PR
    ],

    "Feature Count": [
        303,
        X_train_inner_transformed.shape[1]
    ],

    "Best Iteration": [
        np.nan,
        reference_xgb.best_iteration
    ]
})

display(
    model12_results.style.format({
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
        "ROC-AUC Change": "{:+.4f}",
        "PR-AUC Change": "{:+.4f}"
    })
)

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change,Feature Count,Best Iteration
0,Model11 Optuna,0.7882,0.2905,+0.0000,+0.0000,303,nan
1,Model12 Reference-Inspired,0.7942,0.2975,+0.0060,+0.0070,576,5178.000000


In [40]:
!pip install mlflow -q

import mlflow

mlflow.set_tracking_uri(
    "sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db"
)

mlflow.set_experiment(
    "RupeeRisk"
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787393296537, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787393296537, lifecycle_stage='active', name='RupeeRisk', tags={}, trace_location=None, workspace='default'>

In [43]:
MODEL12_RESULTS_PATH = (
    RESULTS_PATH +
    "model12_reference_inspired_results.csv"
)

model12_results.to_csv(
    MODEL12_RESULTS_PATH,
    index=False
)

print(
    "Model12 results saved:",
    MODEL12_RESULTS_PATH
)

Model12 results saved: /content/drive/MyDrive/RupeeRisk/model12_reference_inspired_results.csv


In [44]:
import joblib

MODEL12_PIPELINE_PATH = (
    RESULTS_PATH +
    "rupeerisk_model12_reference_inspired_xgb.joblib"
)

model12_bundle = {
    "preprocessor": preprocessor,
    "model": reference_xgb,
    "best_iteration": int(reference_xgb.best_iteration),
    "transformed_feature_count": X_train_inner_transformed.shape[1],
    "validation_roc_auc": model12_roc_auc,
    "validation_pr_auc": model12_pr_auc,
    "base_benchmark_roc_auc": 0.7882,
    "base_benchmark_pr_auc": 0.2905
}

joblib.dump(
    model12_bundle,
    MODEL12_PIPELINE_PATH
)

print(
    "Model12 saved:",
    MODEL12_PIPELINE_PATH
)

Model12 saved: /content/drive/MyDrive/RupeeRisk/rupeerisk_model12_reference_inspired_xgb.joblib
